### Realsense d405 카메라 OBB

### cpu 버전

In [18]:
from __future__ import annotations
from typing import Optional, Dict, Any, Tuple
from dataclasses import dataclass
import time
import datetime
import numpy as np
import cv2
import pyrealsense2 as rs
from ultralytics import YOLO

# -----------------------------
# Configuration
# -----------------------------
@dataclass(frozen=True)
class VisionConfig:
    # model
    model_path: str
    conf_thres: float = 0.85
    iou_thres: float = 0.75
    imgsz: int = 640

    # camera stream
    width: int = 640
    height: int = 480
    fps: int = 30

    # sampling (평균낼 횟수)
    avg_n: int = 10
    
    # depth ROI (meters)
    roi_margin_px: float = 6.0
    min_roi_pixels: int = 120
    mad_thres_m: float = 0.030 
    depth_min_m: float = 0.15
    depth_max_m: float = 3.00

    # sanity filters
    z_range_mm: Tuple[float, float] = (150.0, 1200.0)
    
    jump_xy_mm: float = 35.0
    jump_z_mm: float = 60.0
    jump_ang_deg: float = 10.0

    max_consec_skips_reset: int = 15

    # preview / overlay
    preview_win_name: str = "OBB Vision Control (CPU TEST MODE)"
    overlay_font_scale: float = 0.6
    overlay_thickness: int = 2


# 기본 설정 인스턴스
DEFAULT_VISION_CONFIG = VisionConfig(
    model_path=r"/home/dw/ws_job_msislab/amr_project/src/job_pc/runs/obb/20251231_obb_test/weights/best.pt"
)


# -----------------------------
# Local helpers
# -----------------------------
def clamp(v, lo, hi):
    return max(lo, min(hi, v))


def poly_shrink_towards_center(poly4x2: np.ndarray, margin_px: float):
    p = poly4x2.astype(np.float32)
    c = p.mean(axis=0, keepdims=True)
    v = p - c
    norm = np.linalg.norm(v, axis=1, keepdims=True) + 1e-6
    return p - (v / norm) * margin_px


def depth_roi_stats(depth_u16: np.ndarray, depth_scale: float, poly4x2: np.ndarray, cfg: VisionConfig):
    h, w = depth_u16.shape[:2]
    poly = np.round(poly4x2).astype(np.int32)

    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.fillPoly(mask, [poly.reshape(-1, 1, 2)], 255)

    d = depth_u16[mask == 255].astype(np.float32) * depth_scale
    d = d[(d > 0) & (d >= cfg.depth_min_m) & (d <= cfg.depth_max_m)]
    
    if d.size == 0:
        return 0.0, 0.0, 0

    med = float(np.median(d))
    mad = float(np.median(np.abs(d - med)))
    return med, mad, int(d.size)


def XY_from_pixel_and_Z(cx: int, cy: int, intr, Z_m: float):
    X = (cx - intr.ppx) / intr.fx * Z_m
    Y = (cy - intr.ppy) / intr.fy * Z_m
    return float(X), float(Y)


def obb_angle_deg_upright0_rightplus(poly4x2: np.ndarray) -> float:
    p = poly4x2.astype(np.float32)
    c = p.mean(axis=0, keepdims=True)
    q = p - c
    cov = np.cov(q.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    v = eigvecs[:, np.argmax(eigvals)].astype(np.float32)

    vx, vy = float(v[0]), float(v[1])
    if vy < 0:
        vx, vy = -vx, -vy

    angle = float(np.degrees(np.arctan2(vx, vy)))
    angle = -angle
    return angle


def is_jump(prev, cur, cfg: VisionConfig):
    if prev is None:
        return False
    if abs(cur["Xmm"] - prev["Xmm"]) > cfg.jump_xy_mm:
        return True
    if abs(cur["Ymm"] - prev["Ymm"]) > cfg.jump_xy_mm:
        return True
    if abs(cur["Zmm"] - prev["Zmm"]) > cfg.jump_z_mm:
        return True
    if abs(cur["angle"] - prev["angle"]) > cfg.jump_ang_deg:
        return True
    return False


def draw_overlay_xyz_angle(img, Xmm, Ymm, Zmm, angle, cfg: VisionConfig, status_text="", color=(0, 255, 255)):
    line1 = f"cam X {Xmm:+.1f}  Y {Ymm:+.1f}  Z {Zmm:+.1f}  (mm)"
    line2 = f"angle {angle:+.2f} deg"

    x, y = 10, 14
    font = cv2.FONT_HERSHEY_SIMPLEX
    fs = cfg.overlay_font_scale
    th = cfg.overlay_thickness

    (w1, h1), _ = cv2.getTextSize(line1, font, fs, th)
    (w2, h2), _ = cv2.getTextSize(line2, font, fs, th)
    w = max(w1, w2)
    h = h1 + h2 + 18

    overlay = img.copy()
    cv2.rectangle(overlay, (6, 6), (6 + w + 12, 6 + h), (0, 0, 0), -1)
    
    if status_text:
        cv2.rectangle(overlay, (6, img.shape[0] - 40), (350, img.shape[0] - 10), (0, 0, 0), -1)

    cv2.addWeighted(overlay, 0.35, img, 0.65, 0, img)

    cv2.putText(img, line1, (x, y + 18), font, fs, (255, 255, 255), th, cv2.LINE_AA)
    cv2.putText(img, line2, (x, y + 18 + h1 + 6), font, fs, (255, 255, 255), th, cv2.LINE_AA)
    
    if status_text:
        cv2.putText(img, status_text, (10, img.shape[0] - 20), font, fs, color, th, cv2.LINE_AA)


def run_interactive_measurement(cfg: VisionConfig = DEFAULT_VISION_CONFIG):
    print("Loading YOLO model...")
    model = YOLO(cfg.model_path)
    print("Model loaded.")

    print("Initializing RealSense...")
    pipeline = rs.pipeline()
    rs_cfg = rs.config()
    rs_cfg.enable_stream(rs.stream.color, cfg.width, cfg.height, rs.format.bgr8, cfg.fps)
    rs_cfg.enable_stream(rs.stream.depth, cfg.width, cfg.height, rs.format.z16, cfg.fps)

    profile = pipeline.start(rs_cfg)
    align = rs.align(rs.stream.color)

    depth_sensor = profile.get_device().first_depth_sensor()
    depth_scale = float(depth_sensor.get_depth_scale())

    # Filters
    temporal = rs.temporal_filter()
    spatial = rs.spatial_filter()
    hole = rs.hole_filling_filter()
    spatial.set_option(rs.option.filter_magnitude, 2)
    spatial.set_option(rs.option.filter_smooth_alpha, 0.5)
    spatial.set_option(rs.option.filter_smooth_delta, 20)

    # Window Init
    cv2.namedWindow(cfg.preview_win_name, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(cfg.preview_win_name, cfg.width, cfg.height)
    
    # State Variables
    prev_valid = None
    consec_skips = 0
    
    # Data Collection State
    is_collecting = False
    collected_samples = [] 
    
    # --- FPS Calculation Variables ---
    prev_time = 0
    curr_time = 0
    
    print("\n-------------------------------------------")
    print(" [ CPU TEST MODE START ]")
    print("  'm' : Measure")
    print("  's' : Screenshot")
    print("  'q' : Quit")
    print("-------------------------------------------\n")

    try:
        while True:
            # 1. 시간 측정 시작
            curr_time = time.time()
            
            frames = pipeline.wait_for_frames()
            frames = align.process(frames)

            color_frame = frames.get_color_frame()
            depth_frame = frames.get_depth_frame()
            if not color_frame or not depth_frame:
                continue

            # Process Depth
            depth_frame = spatial.process(depth_frame).as_depth_frame()
            depth_frame = temporal.process(depth_frame).as_depth_frame()
            depth_frame = hole.process(depth_frame).as_depth_frame()

            frame = np.asanyarray(color_frame.get_data())
            intr = color_frame.profile.as_video_stream_profile().get_intrinsics()
            depth_u16 = np.asanyarray(depth_frame.get_data())
            
            vis = frame.copy()
            
            # 2. Predict (CPU 강제 설정)
            # device='cpu' 옵션 추가됨
            results = model.predict(
                frame, 
                imgsz=cfg.imgsz, 
                conf=cfg.conf_thres, 
                iou=cfg.iou_thres, 
                verbose=False,
                device='cpu' 
            )
            r = results[0]

            candidates = []
            img_cx = (cfg.width - 1) * 0.5
            img_cy = (cfg.height - 1) * 0.5

            if getattr(r, "obb", None) is not None and r.obb is not None:
                obb = r.obb
                if obb.xyxyxyxy is not None and len(obb.xyxyxyxy) > 0:
                    polys = obb.xyxyxyxy.cpu().numpy()
                    confs = obb.conf.cpu().numpy().astype(float)
                    clss  = obb.cls.cpu().numpy().astype(int)

                    for poly8, cf, ci in zip(polys, confs, clss):
                        if float(cf) < cfg.conf_thres:
                            continue
                        poly = poly8.reshape(4, 2)
                        cx_det = float(np.mean(poly[:, 0]))
                        cy_det = float(np.mean(poly[:, 1]))
                        dx = cx_det - img_cx
                        dy = cy_det - img_cy
                        dist2 = dx * dx + dy * dy
                        candidates.append((dist2, -float(cf), float(cf), int(ci), poly, cx_det, cy_det))

            current_valid_sample = None
            
            if candidates:
                candidates.sort()
                dist2, _ncf, cf, ci, poly, cx_det_f, cy_det_f = candidates[0]
                
                cx = clamp(int(round(cx_det_f)), 0, cfg.width - 1)
                cy = clamp(int(round(cy_det_f)), 0, cfg.height - 1)

                poly_i = np.round(poly).astype(np.int32).reshape(-1, 1, 2)
                cv2.polylines(vis, [poly_i], True, (0, 255, 0), 2)
                cv2.circle(vis, (cx, cy), 5, (0, 0, 255), -1)

                poly_shrunk = poly_shrink_towards_center(poly, cfg.roi_margin_px)
                poly_shrunk[:, 0] = np.clip(poly_shrunk[:, 0], 0, cfg.width - 1)
                poly_shrunk[:, 1] = np.clip(poly_shrunk[:, 1], 0, cfg.height - 1)

                Z_roi_m, mad_m, roi_n = depth_roi_stats(depth_u16, depth_scale, poly_shrunk, cfg)
                
                depth_ok = (Z_roi_m > 0.0 and roi_n >= cfg.min_roi_pixels and mad_m <= cfg.mad_thres_m)
                
                if depth_ok:
                    Z_use_m = Z_roi_m 
                    Z_use_mm = Z_use_m * 1000.0
                    
                    if (cfg.z_range_mm[0] <= Z_use_mm <= cfg.z_range_mm[1]):
                        X_m, Y_m = XY_from_pixel_and_Z(cx, cy, intr, Z_use_m)
                        angle = obb_angle_deg_upright0_rightplus(poly)
                        
                        cur = {
                            "Xmm": X_m * 1000.0,
                            "Ymm": Y_m * 1000.0,
                            "Zmm": Z_use_m * 1000.0,
                            "angle": float(angle),
                        }
                        
                        if not is_jump(prev_valid, cur, cfg):
                            current_valid_sample = cur
                            prev_valid = cur
                            consec_skips = 0
                        else:
                            consec_skips += 1
                    else:
                        consec_skips += 1
                else:
                    consec_skips += 1

                if consec_skips >= cfg.max_consec_skips_reset:
                    prev_valid = None
                    consec_skips = 0

            else:
                consec_skips += 1
                if consec_skips >= cfg.max_consec_skips_reset:
                    prev_valid = None
                    consec_skips = 0

            # 3. FPS 계산 및 표시
            fps = 1.0 / (curr_time - prev_time) if prev_time != 0 else 0
            prev_time = curr_time
            
            cv2.putText(vis, f"CPU MODE - FPS: {fps:.1f}", (10, 40), 
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 2)

            # Data Collection
            if is_collecting:
                status_msg = f"[Collecting: {len(collected_samples)}/{cfg.avg_n}]"
                status_color = (0, 0, 255) 
                
                if current_valid_sample is not None:
                    collected_samples.append([
                        current_valid_sample["Xmm"],
                        current_valid_sample["Ymm"],
                        current_valid_sample["Zmm"],
                        current_valid_sample["angle"]
                    ])
                
                if len(collected_samples) >= cfg.avg_n:
                    arr = np.array(collected_samples, dtype=np.float32)
                    mean_val = np.mean(arr, axis=0)
                    
                    print("\n" + "="*40)
                    print(f" [MEASUREMENT RESULT (Avg of {cfg.avg_n})]")
                    print(f"  X : {mean_val[0]:.1f} mm")
                    print(f"  Y : {mean_val[1]:.1f} mm")
                    print(f"  Z : {mean_val[2]:.1f} mm")
                    print(f"  A : {mean_val[3]:.2f} deg")
                    print("="*40 + "\n")
                    
                    is_collecting = False
                    collected_samples = []

            else:
                status_msg = "[Monitor Mode] Press 'm' to Measure"
                status_color = (0, 255, 0)

            if prev_valid is not None:
                draw_overlay_xyz_angle(vis, 
                                       prev_valid["Xmm"], 
                                       prev_valid["Ymm"], 
                                       prev_valid["Zmm"], 
                                       prev_valid["angle"], 
                                       cfg, 
                                       status_msg, 
                                       status_color)
            else:
                 cv2.putText(vis, status_msg, (10, vis.shape[0] - 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, status_color, 2, cv2.LINE_AA)

            cv2.imshow(cfg.preview_win_name, vis)
            
            key = cv2.waitKey(1) & 0xFF
            
            if key == 27 or key == ord('q'):
                break
            
            if key == ord('m'): 
                if not is_collecting:
                    print(f"Start collecting {cfg.avg_n} samples...")
                    is_collecting = True
                    collected_samples = []
                else:
                    print("Already collecting... please wait.")

            if key == ord('s'): 
                ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
                filename = f"screenshot_{ts}.png"
                cv2.imwrite(filename, vis)
                print(f"Screenshot saved: {filename}")

    finally:
        try:
            pipeline.stop()
        except Exception:
            pass
        try:
            cv2.destroyAllWindows()
        except Exception:
            pass

if __name__ == "__main__":
    run_interactive_measurement(DEFAULT_VISION_CONFIG)

Loading YOLO model...
Model loaded.
Initializing RealSense...

-------------------------------------------
 [ CPU TEST MODE START ]
  'm' : Measure
  's' : Screenshot
  'q' : Quit
-------------------------------------------

Start collecting 10 samples...

 [MEASUREMENT RESULT (Avg of 10)]
  X : 20.4 mm
  Y : 29.8 mm
  Z : 383.8 mm
  A : -6.20 deg



### gpu 버전

In [ ]:
from __future__ import annotations
from typing import Optional, Dict, Any, Tuple
from dataclasses import dataclass
import time
import datetime
import numpy as np
import cv2
import pyrealsense2 as rs
from ultralytics import YOLO

# -----------------------------
# Configuration
# -----------------------------
@dataclass(frozen=True)
class VisionConfig:
    # model
    model_path: str
    conf_thres: float = 0.85
    iou_thres: float = 0.75
    imgsz: int = 640

    # camera stream
    width: int = 640
    height: int = 480
    fps: int = 30

    # sampling (평균낼 횟수)
    avg_n: int = 10
    
    # depth ROI (meters)
    roi_margin_px: float = 6.0
    min_roi_pixels: int = 120
    mad_thres_m: float = 0.030 # 센서값만 쓰므로 허용 오차를 살짝 늘림 (0.02 -> 0.03)
    depth_min_m: float = 0.15
    depth_max_m: float = 3.00

    # sanity filters
    z_range_mm: Tuple[float, float] = (150.0, 1200.0)
    
    jump_xy_mm: float = 35.0
    jump_z_mm: float = 60.0
    jump_ang_deg: float = 10.0

    max_consec_skips_reset: int = 15

    # preview / overlay
    preview_win_name: str = "OBB Vision Control (Depth Sensor Only)"
    overlay_font_scale: float = 0.6
    overlay_thickness: int = 2


# 기본 설정 인스턴스
DEFAULT_VISION_CONFIG = VisionConfig(
    # 모델 경로 확인 필수
    model_path=r"/home/dw/ws_job_msislab/amr_project/src/job_pc/runs/obb/20251231_obb_test/weights/best.pt"
)

# -----------------------------
# Local helpers
# -----------------------------
def clamp(v, lo, hi):
    return max(lo, min(hi, v))


def poly_shrink_towards_center(poly4x2: np.ndarray, margin_px: float):
    """ 박스 테두리 노이즈를 피하기 위해 폴리곤을 안쪽으로 축소 """
    p = poly4x2.astype(np.float32)
    c = p.mean(axis=0, keepdims=True)
    v = p - c
    norm = np.linalg.norm(v, axis=1, keepdims=True) + 1e-6
    return p - (v / norm) * margin_px


def depth_roi_stats(depth_u16: np.ndarray, depth_scale: float, poly4x2: np.ndarray, cfg: VisionConfig):
    """ 폴리곤 내부의 Depth 값들의 중앙값(Median)과 편차(MAD)를 구함 """
    h, w = depth_u16.shape[:2]
    poly = np.round(poly4x2).astype(np.int32)

    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.fillPoly(mask, [poly.reshape(-1, 1, 2)], 255)

    # 마스크 영역의 깊이 값 추출
    d = depth_u16[mask == 255].astype(np.float32) * depth_scale
    
    # 유효 범위 필터링
    d = d[(d > 0) & (d >= cfg.depth_min_m) & (d <= cfg.depth_max_m)]
    
    if d.size == 0:
        return 0.0, 0.0, 0

    med = float(np.median(d))
    mad = float(np.median(np.abs(d - med)))
    return med, mad, int(d.size)


def XY_from_pixel_and_Z(cx: int, cy: int, intr, Z_m: float):
    """ Pinhole Model로 3D 좌표 계산 """
    X = (cx - intr.ppx) / intr.fx * Z_m
    Y = (cy - intr.ppy) / intr.fy * Z_m
    return float(X), float(Y)  # meters


def obb_angle_deg_upright0_rightplus(poly4x2: np.ndarray) -> float:
    """ PCA를 이용한 회전 각도 계산 """
    p = poly4x2.astype(np.float32)
    c = p.mean(axis=0, keepdims=True)
    q = p - c
    cov = np.cov(q.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    v = eigvecs[:, np.argmax(eigvals)].astype(np.float32)

    vx, vy = float(v[0]), float(v[1])
    if vy < 0:
        vx, vy = -vx, -vy

    angle = float(np.degrees(np.arctan2(vx, vy)))
    angle = -angle
    return angle


def is_jump(prev, cur, cfg: VisionConfig):
    """ 값이 갑자기 튀는지 검사 """
    if prev is None:
        return False
    if abs(cur["Xmm"] - prev["Xmm"]) > cfg.jump_xy_mm:
        return True
    if abs(cur["Ymm"] - prev["Ymm"]) > cfg.jump_xy_mm:
        return True
    if abs(cur["Zmm"] - prev["Zmm"]) > cfg.jump_z_mm:
        return True
    if abs(cur["angle"] - prev["angle"]) > cfg.jump_ang_deg:
        return True
    return False


def draw_overlay_xyz_angle(img, Xmm, Ymm, Zmm, angle, cfg: VisionConfig, status_text="", color=(0, 255, 255)):
    line1 = f"cam X {Xmm:+.1f}  Y {Ymm:+.1f}  Z {Zmm:+.1f}  (mm)"
    line2 = f"angle {angle:+.2f} deg"

    x, y = 10, 14
    font = cv2.FONT_HERSHEY_SIMPLEX
    fs = cfg.overlay_font_scale
    th = cfg.overlay_thickness

    (w1, h1), _ = cv2.getTextSize(line1, font, fs, th)
    (w2, h2), _ = cv2.getTextSize(line2, font, fs, th)
    w = max(w1, w2)
    h = h1 + h2 + 18

    overlay = img.copy()
    cv2.rectangle(overlay, (6, 6), (6 + w + 12, 6 + h), (0, 0, 0), -1)
    
    # Bottom Status Bar
    if status_text:
        cv2.rectangle(overlay, (6, img.shape[0] - 40), (350, img.shape[0] - 10), (0, 0, 0), -1)

    cv2.addWeighted(overlay, 0.35, img, 0.65, 0, img)

    cv2.putText(img, line1, (x, y + 18), font, fs, (255, 255, 255), th, cv2.LINE_AA)
    cv2.putText(img, line2, (x, y + 18 + h1 + 6), font, fs, (255, 255, 255), th, cv2.LINE_AA)
    
    if status_text:
        cv2.putText(img, status_text, (10, img.shape[0] - 20), font, fs, color, th, cv2.LINE_AA)


def run_interactive_measurement(cfg: VisionConfig = DEFAULT_VISION_CONFIG):
    print("Loading YOLO model...")
    model = YOLO(cfg.model_path)
    print("Model loaded.")

    print("Initializing RealSense...")
    pipeline = rs.pipeline()
    rs_cfg = rs.config()
    rs_cfg.enable_stream(rs.stream.color, cfg.width, cfg.height, rs.format.bgr8, cfg.fps)
    rs_cfg.enable_stream(rs.stream.depth, cfg.width, cfg.height, rs.format.z16, cfg.fps)

    profile = pipeline.start(rs_cfg)
    align = rs.align(rs.stream.color)

    depth_sensor = profile.get_device().first_depth_sensor()
    depth_scale = float(depth_sensor.get_depth_scale())

    # Filters (Depth 품질 향상을 위해 켜둠)
    temporal = rs.temporal_filter()
    spatial = rs.spatial_filter()
    hole = rs.hole_filling_filter()
    spatial.set_option(rs.option.filter_magnitude, 2)
    spatial.set_option(rs.option.filter_smooth_alpha, 0.5)
    spatial.set_option(rs.option.filter_smooth_delta, 20)

    # Window Init
    cv2.namedWindow(cfg.preview_win_name, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(cfg.preview_win_name, cfg.width, cfg.height)
    
    # State Variables
    prev_valid = None
    consec_skips = 0
    
    # Data Collection State
    is_collecting = False
    collected_samples = [] 
    
    print("\n-------------------------------------------")
    print(" [ Controls ]")
    print(f"  'm' : Trigger Measurement (Collect {cfg.avg_n} samples & Print)")
    print("  's' : Save Screenshot")
    print("  'q' / ESC : Quit")
    print("-------------------------------------------\n")

    try:
        while True:
            frames = pipeline.wait_for_frames()
            frames = align.process(frames)

            color_frame = frames.get_color_frame()
            depth_frame = frames.get_depth_frame()
            if not color_frame or not depth_frame:
                continue

            # Process Depth
            depth_frame = spatial.process(depth_frame).as_depth_frame()
            depth_frame = temporal.process(depth_frame).as_depth_frame()
            depth_frame = hole.process(depth_frame).as_depth_frame()

            frame = np.asanyarray(color_frame.get_data())
            intr = color_frame.profile.as_video_stream_profile().get_intrinsics()
            depth_u16 = np.asanyarray(depth_frame.get_data())
            
            vis = frame.copy()
            
            # Predict
            results = model.predict(frame, imgsz=cfg.imgsz, conf=cfg.conf_thres, iou=cfg.iou_thres, verbose=False)
            r = results[0]

            candidates = []
            img_cx = (cfg.width - 1) * 0.5
            img_cy = (cfg.height - 1) * 0.5

            if getattr(r, "obb", None) is not None and r.obb is not None:
                obb = r.obb
                if obb.xyxyxyxy is not None and len(obb.xyxyxyxy) > 0:
                    polys = obb.xyxyxyxy.cpu().numpy()
                    confs = obb.conf.cpu().numpy().astype(float)
                    clss  = obb.cls.cpu().numpy().astype(int)

                    for poly8, cf, ci in zip(polys, confs, clss):
                        if float(cf) < cfg.conf_thres:
                            continue
                        poly = poly8.reshape(4, 2)
                        cx_det = float(np.mean(poly[:, 0]))
                        cy_det = float(np.mean(poly[:, 1]))
                        dx = cx_det - img_cx
                        dy = cy_det - img_cy
                        dist2 = dx * dx + dy * dy
                        candidates.append((dist2, -float(cf), float(cf), int(ci), poly, cx_det, cy_det))

            # --- Logic to determine current frame status ---
            current_valid_sample = None
            
            if candidates:
                candidates.sort()
                dist2, _ncf, cf, ci, poly, cx_det_f, cy_det_f = candidates[0]
                
                cx = clamp(int(round(cx_det_f)), 0, cfg.width - 1)
                cy = clamp(int(round(cy_det_f)), 0, cfg.height - 1)

                # Draw Visuals
                poly_i = np.round(poly).astype(np.int32).reshape(-1, 1, 2)
                cv2.polylines(vis, [poly_i], True, (0, 255, 0), 2)
                cv2.circle(vis, (cx, cy), 5, (0, 0, 255), -1)

                # Depth Calculation (Sensor Only)
                poly_shrunk = poly_shrink_towards_center(poly, cfg.roi_margin_px)
                poly_shrunk[:, 0] = np.clip(poly_shrunk[:, 0], 0, cfg.width - 1)
                poly_shrunk[:, 1] = np.clip(poly_shrunk[:, 1], 0, cfg.height - 1)

                # ROI 내부의 중앙값(Median) 추출
                Z_roi_m, mad_m, roi_n = depth_roi_stats(depth_u16, depth_scale, poly_shrunk, cfg)
                
                # 유효성 검사 (픽셀 개수, MAD 편차)
                depth_ok = (Z_roi_m > 0.0 and roi_n >= cfg.min_roi_pixels and mad_m <= cfg.mad_thres_m)
                
                if depth_ok:
                    Z_use_m = Z_roi_m # 순수 센서값 사용
                    Z_use_mm = Z_use_m * 1000.0
                    
                    # Range Check
                    if (cfg.z_range_mm[0] <= Z_use_mm <= cfg.z_range_mm[1]):
                        X_m, Y_m = XY_from_pixel_and_Z(cx, cy, intr, Z_use_m)
                        angle = obb_angle_deg_upright0_rightplus(poly)
                        
                        cur = {
                            "Xmm": X_m * 1000.0,
                            "Ymm": Y_m * 1000.0,
                            "Zmm": Z_use_m * 1000.0,
                            "angle": float(angle),
                        }
                        
                        # Jump Filter (값 튐 방지)
                        if not is_jump(prev_valid, cur, cfg):
                            current_valid_sample = cur
                            prev_valid = cur
                            consec_skips = 0
                        else:
                            consec_skips += 1
                    else:
                        consec_skips += 1
                else:
                    # Depth가 너무 튀거나 데이터가 없으면 Skip
                    consec_skips += 1

                # 리셋 로직
                if consec_skips >= cfg.max_consec_skips_reset:
                    prev_valid = None
                    consec_skips = 0

            else:
                consec_skips += 1
                if consec_skips >= cfg.max_consec_skips_reset:
                    prev_valid = None
                    consec_skips = 0

            # --- Data Collection Logic ---
            if is_collecting:
                status_msg = f"[Collecting: {len(collected_samples)}/{cfg.avg_n}]"
                status_color = (0, 0, 255) # Red while collecting
                
                if current_valid_sample is not None:
                    collected_samples.append([
                        current_valid_sample["Xmm"],
                        current_valid_sample["Ymm"],
                        current_valid_sample["Zmm"],
                        current_valid_sample["angle"]
                    ])
                
                if len(collected_samples) >= cfg.avg_n:
                    arr = np.array(collected_samples, dtype=np.float32)
                    mean_val = np.mean(arr, axis=0)
                    
                    print("\n" + "="*40)
                    print(f" [MEASUREMENT RESULT (Avg of {cfg.avg_n})]")
                    print(f"  X : {mean_val[0]:.1f} mm")
                    print(f"  Y : {mean_val[1]:.1f} mm")
                    print(f"  Z : {mean_val[2]:.1f} mm")
                    print(f"  A : {mean_val[3]:.2f} deg")
                    print("="*40 + "\n")
                    
                    is_collecting = False
                    collected_samples = []

            else:
                status_msg = "[Monitor Mode] Press 'm' to Measure"
                status_color = (0, 255, 0) # Green while waiting

            # --- Update Display ---
            if prev_valid is not None:
                draw_overlay_xyz_angle(vis, 
                                       prev_valid["Xmm"], 
                                       prev_valid["Ymm"], 
                                       prev_valid["Zmm"], 
                                       prev_valid["angle"], 
                                       cfg, 
                                       status_msg, 
                                       status_color)
            else:
                 cv2.putText(vis, status_msg, (10, vis.shape[0] - 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, status_color, 2, cv2.LINE_AA)

            cv2.imshow(cfg.preview_win_name, vis)
            
            key = cv2.waitKey(1) & 0xFF
            if key == 27 or key == ord('q'):
                break
            if key == ord('m'):
                if not is_collecting:
                    print(f"Start collecting {cfg.avg_n} samples...")
                    is_collecting = True
                    collected_samples = []
                else:
                    print("Already collecting...")
            if key == ord('s'):
                ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
                cv2.imwrite(f"screenshot_{ts}.png", vis)
                print("Screenshot saved.")

    finally:
        try: pipeline.stop()
        except: pass
        try: cv2.destroyAllWindows()
        except: pass

if __name__ == "__main__":
    run_interactive_measurement(DEFAULT_VISION_CONFIG)

Loading YOLO model...
Model loaded.
Initializing RealSense...

-------------------------------------------
 [ Controls ]
  'm' : Trigger Measurement (Collect 10 samples & Print)
  's' : Save Screenshot
  'q' / ESC : Quit
-------------------------------------------



### 박스와의 거리도 표시

In [21]:
from __future__ import annotations
from typing import Optional, Dict, Any, Tuple
from dataclasses import dataclass
import time
import datetime
import numpy as np
import cv2
import pyrealsense2 as rs
from ultralytics import YOLO

# -----------------------------
# Configuration
# -----------------------------
@dataclass(frozen=True)
class VisionConfig:
    # model
    model_path: str
    conf_thres: float = 0.95
    iou_thres: float = 0.90
    imgsz: int = 640

    # camera stream
    width: int = 640
    height: int = 480
    fps: int = 30

    # sampling (평균낼 횟수)
    avg_n: int = 10
    
    # depth ROI (meters)
    roi_margin_px: float = 6.0
    min_roi_pixels: int = 120
    mad_thres_m: float = 0.030 
    depth_min_m: float = 0.15
    depth_max_m: float = 3.00

    # sanity filters
    z_range_mm: Tuple[float, float] = (150.0, 1200.0)
    
    jump_xy_mm: float = 35.0
    jump_z_mm: float = 60.0
    jump_ang_deg: float = 10.0

    max_consec_skips_reset: int = 15

    # preview / overlay
    preview_win_name: str = "OBB Vision Control (Depth Sensor Only)"
    overlay_font_scale: float = 0.6
    overlay_thickness: int = 2


# 기본 설정 인스턴스
DEFAULT_VISION_CONFIG = VisionConfig(
    # 모델 경로 확인 필수
    model_path=r"/home/dw/ws_job_msislab/amr_project/src/job_pc/runs/obb/20251231_obb_test/weights/best.pt"
)


# -----------------------------
# Local helpers
# -----------------------------
def clamp(v, lo, hi):
    return max(lo, min(hi, v))


def poly_shrink_towards_center(poly4x2: np.ndarray, margin_px: float):
    """ 박스 테두리 노이즈를 피하기 위해 폴리곤을 안쪽으로 축소 """
    p = poly4x2.astype(np.float32)
    c = p.mean(axis=0, keepdims=True)
    v = p - c
    norm = np.linalg.norm(v, axis=1, keepdims=True) + 1e-6
    return p - (v / norm) * margin_px


def depth_roi_stats(depth_u16: np.ndarray, depth_scale: float, poly4x2: np.ndarray, cfg: VisionConfig):
    """ 폴리곤 내부의 Depth 값들의 중앙값(Median)과 편차(MAD)를 구함 """
    h, w = depth_u16.shape[:2]
    poly = np.round(poly4x2).astype(np.int32)

    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.fillPoly(mask, [poly.reshape(-1, 1, 2)], 255)

    # 마스크 영역의 깊이 값 추출
    d = depth_u16[mask == 255].astype(np.float32) * depth_scale
    
    # 유효 범위 필터링
    d = d[(d > 0) & (d >= cfg.depth_min_m) & (d <= cfg.depth_max_m)]
    
    if d.size == 0:
        return 0.0, 0.0, 0

    med = float(np.median(d))
    mad = float(np.median(np.abs(d - med)))
    return med, mad, int(d.size)


def XY_from_pixel_and_Z(cx: int, cy: int, intr, Z_m: float):
    """ Pinhole Model로 3D 좌표 계산 """
    X = (cx - intr.ppx) / intr.fx * Z_m
    Y = (cy - intr.ppy) / intr.fy * Z_m
    return float(X), float(Y)  # meters


def obb_angle_deg_upright0_rightplus(poly4x2: np.ndarray) -> float:
    """ PCA를 이용한 회전 각도 계산 """
    p = poly4x2.astype(np.float32)
    c = p.mean(axis=0, keepdims=True)
    q = p - c
    cov = np.cov(q.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    v = eigvecs[:, np.argmax(eigvals)].astype(np.float32)

    vx, vy = float(v[0]), float(v[1])
    if vy < 0:
        vx, vy = -vx, -vy

    angle = float(np.degrees(np.arctan2(vx, vy)))
    angle = -angle
    return angle


def is_jump(prev, cur, cfg: VisionConfig):
    """ 값이 갑자기 튀는지 검사 """
    if prev is None:
        return False
    if abs(cur["Xmm"] - prev["Xmm"]) > cfg.jump_xy_mm:
        return True
    if abs(cur["Ymm"] - prev["Ymm"]) > cfg.jump_xy_mm:
        return True
    if abs(cur["Zmm"] - prev["Zmm"]) > cfg.jump_z_mm:
        return True
    if abs(cur["angle"] - prev["angle"]) > cfg.jump_ang_deg:
        return True
    return False


def draw_overlay_xyz_angle(img, Xmm, Ymm, Zmm, Dist_mm, angle, cfg: VisionConfig, status_text="", color=(0, 255, 255)):
    # 
    line1 = f"cam X {Xmm:+.1f}  Y {Ymm:+.1f}  Z {Zmm:+.1f}"
    line2 = f"Dist {Dist_mm:.1f} mm  |  Angle {angle:+.2f} deg" # 거리를 두 번째 줄에 추가

    x, y = 10, 14
    font = cv2.FONT_HERSHEY_SIMPLEX
    fs = cfg.overlay_font_scale
    th = cfg.overlay_thickness

    (w1, h1), _ = cv2.getTextSize(line1, font, fs, th)
    (w2, h2), _ = cv2.getTextSize(line2, font, fs, th)
    w = max(w1, w2)
    h = h1 + h2 + 18

    overlay = img.copy()
    cv2.rectangle(overlay, (6, 6), (6 + w + 12, 6 + h), (0, 0, 0), -1)
    
    # Bottom Status Bar
    if status_text:
        cv2.rectangle(overlay, (6, img.shape[0] - 40), (350, img.shape[0] - 10), (0, 0, 0), -1)

    cv2.addWeighted(overlay, 0.35, img, 0.65, 0, img)

    cv2.putText(img, line1, (x, y + 18), font, fs, (255, 255, 255), th, cv2.LINE_AA)
    cv2.putText(img, line2, (x, y + 18 + h1 + 6), font, fs, (255, 255, 255), th, cv2.LINE_AA)
    
    if status_text:
        cv2.putText(img, status_text, (10, img.shape[0] - 20), font, fs, color, th, cv2.LINE_AA)


def run_interactive_measurement(cfg: VisionConfig = DEFAULT_VISION_CONFIG):
    print("Loading YOLO model...")
    model = YOLO(cfg.model_path)
    print("Model loaded.")

    print("Initializing RealSense...")
    pipeline = rs.pipeline()
    rs_cfg = rs.config()
    rs_cfg.enable_stream(rs.stream.color, cfg.width, cfg.height, rs.format.bgr8, cfg.fps)
    rs_cfg.enable_stream(rs.stream.depth, cfg.width, cfg.height, rs.format.z16, cfg.fps)

    profile = pipeline.start(rs_cfg)
    align = rs.align(rs.stream.color)

    depth_sensor = profile.get_device().first_depth_sensor()
    depth_scale = float(depth_sensor.get_depth_scale())

    # Filters
    temporal = rs.temporal_filter()
    spatial = rs.spatial_filter()
    hole = rs.hole_filling_filter()
    spatial.set_option(rs.option.filter_magnitude, 2)
    spatial.set_option(rs.option.filter_smooth_alpha, 0.5)
    spatial.set_option(rs.option.filter_smooth_delta, 20)

    # Window Init
    cv2.namedWindow(cfg.preview_win_name, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(cfg.preview_win_name, cfg.width, cfg.height)
    
    # State Variables
    prev_valid = None
    consec_skips = 0
    
    # Data Collection State
    is_collecting = False
    collected_samples = [] 
    
    print("\n-------------------------------------------")
    print(" [ Controls ]")
    print(f"  'm' : Trigger Measurement (Collect {cfg.avg_n} samples & Print)")
    print("  's' : Save Screenshot")
    print("  'q' / ESC : Quit")
    print("-------------------------------------------\n")

    try:
        while True:
            frames = pipeline.wait_for_frames()
            frames = align.process(frames)

            color_frame = frames.get_color_frame()
            depth_frame = frames.get_depth_frame()
            if not color_frame or not depth_frame:
                continue

            # Process Depth
            depth_frame = spatial.process(depth_frame).as_depth_frame()
            depth_frame = temporal.process(depth_frame).as_depth_frame()
            depth_frame = hole.process(depth_frame).as_depth_frame()

            frame = np.asanyarray(color_frame.get_data())
            intr = color_frame.profile.as_video_stream_profile().get_intrinsics()
            depth_u16 = np.asanyarray(depth_frame.get_data())
            
            vis = frame.copy()
            
            # Predict
            results = model.predict(frame, imgsz=cfg.imgsz, conf=cfg.conf_thres, iou=cfg.iou_thres, verbose=False)
            r = results[0]

            candidates = []
            img_cx = (cfg.width - 1) * 0.5
            img_cy = (cfg.height - 1) * 0.5

            if getattr(r, "obb", None) is not None and r.obb is not None:
                obb = r.obb
                if obb.xyxyxyxy is not None and len(obb.xyxyxyxy) > 0:
                    polys = obb.xyxyxyxy.cpu().numpy()
                    confs = obb.conf.cpu().numpy().astype(float)
                    clss  = obb.cls.cpu().numpy().astype(int)

                    for poly8, cf, ci in zip(polys, confs, clss):
                        if float(cf) < cfg.conf_thres:
                            continue
                        poly = poly8.reshape(4, 2)
                        cx_det = float(np.mean(poly[:, 0]))
                        cy_det = float(np.mean(poly[:, 1]))
                        dx = cx_det - img_cx
                        dy = cy_det - img_cy
                        dist2 = dx * dx + dy * dy
                        candidates.append((dist2, -float(cf), float(cf), int(ci), poly, cx_det, cy_det))

            # --- Logic to determine current frame status ---
            current_valid_sample = None
            
            if candidates:
                candidates.sort()
                dist2, _ncf, cf, ci, poly, cx_det_f, cy_det_f = candidates[0]
                
                cx = clamp(int(round(cx_det_f)), 0, cfg.width - 1)
                cy = clamp(int(round(cy_det_f)), 0, cfg.height - 1)

                # Draw Visuals
                poly_i = np.round(poly).astype(np.int32).reshape(-1, 1, 2)
                cv2.polylines(vis, [poly_i], True, (0, 255, 0), 2)
                cv2.circle(vis, (cx, cy), 5, (0, 0, 255), -1)

                # Depth Calculation (Sensor Only)
                poly_shrunk = poly_shrink_towards_center(poly, cfg.roi_margin_px)
                poly_shrunk[:, 0] = np.clip(poly_shrunk[:, 0], 0, cfg.width - 1)
                poly_shrunk[:, 1] = np.clip(poly_shrunk[:, 1], 0, cfg.height - 1)

                Z_roi_m, mad_m, roi_n = depth_roi_stats(depth_u16, depth_scale, poly_shrunk, cfg)
                
                depth_ok = (Z_roi_m > 0.0 and roi_n >= cfg.min_roi_pixels and mad_m <= cfg.mad_thres_m)
                
                if depth_ok:
                    Z_use_m = Z_roi_m 
                    Z_use_mm = Z_use_m * 1000.0
                    
                    if (cfg.z_range_mm[0] <= Z_use_mm <= cfg.z_range_mm[1]):
                        X_m, Y_m = XY_from_pixel_and_Z(cx, cy, intr, Z_use_m)
                        angle = obb_angle_deg_upright0_rightplus(poly)
                        
                        # [거리 계산 추가]
                        dist_m = np.sqrt(X_m**2 + Y_m**2 + Z_use_m**2)

                        cur = {
                            "Xmm": X_m * 1000.0,
                            "Ymm": Y_m * 1000.0,
                            "Zmm": Z_use_m * 1000.0,
                            "Dist_mm": dist_m * 1000.0, # 거리(mm) 추가
                            "angle": float(angle),
                        }
                        
                        if not is_jump(prev_valid, cur, cfg):
                            current_valid_sample = cur
                            prev_valid = cur
                            consec_skips = 0
                        else:
                            consec_skips += 1
                    else:
                        consec_skips += 1
                else:
                    consec_skips += 1

                if consec_skips >= cfg.max_consec_skips_reset:
                    prev_valid = None
                    consec_skips = 0

            else:
                consec_skips += 1
                if consec_skips >= cfg.max_consec_skips_reset:
                    prev_valid = None
                    consec_skips = 0

            # --- Data Collection Logic ---
            if is_collecting:
                status_msg = f"[Collecting: {len(collected_samples)}/{cfg.avg_n}]"
                status_color = (0, 0, 255) 
                
                if current_valid_sample is not None:
                    collected_samples.append([
                        current_valid_sample["Xmm"],
                        current_valid_sample["Ymm"],
                        current_valid_sample["Zmm"],
                        current_valid_sample["angle"],
                        current_valid_sample["Dist_mm"] # 샘플에 거리 추가
                    ])
                
                if len(collected_samples) >= cfg.avg_n:
                    arr = np.array(collected_samples, dtype=np.float32)
                    mean_val = np.mean(arr, axis=0)
                    
                    print("\n" + "="*40)
                    print(f" [MEASUREMENT RESULT (Avg of {cfg.avg_n})]")
                    print(f"  X    : {mean_val[0]:.1f} mm")
                    print(f"  Y    : {mean_val[1]:.1f} mm")
                    print(f"  Z    : {mean_val[2]:.1f} mm")
                    print(f"  Dist : {mean_val[4]:.1f} mm") # 평균 거리 출력
                    print(f"  A    : {mean_val[3]:.2f} deg")
                    print("="*40 + "\n")
                    
                    is_collecting = False
                    collected_samples = []

            else:
                status_msg = "[Monitor Mode] Press 'm' to Measure"
                status_color = (0, 255, 0)

            # --- Update Display ---
            if prev_valid is not None:
                draw_overlay_xyz_angle(vis, 
                                       prev_valid["Xmm"], 
                                       prev_valid["Ymm"], 
                                       prev_valid["Zmm"], 
                                       prev_valid["Dist_mm"], # 거리 전달
                                       prev_valid["angle"], 
                                       cfg, 
                                       status_msg, 
                                       status_color)
            else:
                 cv2.putText(vis, status_msg, (10, vis.shape[0] - 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, status_color, 2, cv2.LINE_AA)

            cv2.imshow(cfg.preview_win_name, vis)
            
            key = cv2.waitKey(1) & 0xFF
            
            if key == 27 or key == ord('q'):
                break
            
            if key == ord('m'):
                if not is_collecting:
                    print(f"Start collecting {cfg.avg_n} samples...")
                    is_collecting = True
                    collected_samples = []
                else:
                    print("Already collecting... please wait.")

            if key == ord('s'):
                ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
                filename = f"screenshot_{ts}.png"
                cv2.imwrite(filename, vis)
                print(f"Screenshot saved: {filename}")

    finally:
        try:
            pipeline.stop()
        except Exception:
            pass
        try:
            cv2.destroyAllWindows()
        except Exception:
            pass

if __name__ == "__main__":
    run_interactive_measurement(DEFAULT_VISION_CONFIG)

Loading YOLO model...
Model loaded.
Initializing RealSense...

-------------------------------------------
 [ Controls ]
  'm' : Trigger Measurement (Collect 10 samples & Print)
  's' : Save Screenshot
  'q' / ESC : Quit
-------------------------------------------



### Realsense d405 카메라 seg

### Oak 카메라